In [1]:
import json
import os
import numpy as np

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, state_fidelity
from qiskit_ionq import GPI2Gate, MSGate


def ket0():
    return np.array([1.0, 0.0], dtype=complex)


def ket1():
    return np.array([0.0, 1.0], dtype=complex)


def projector(v):
    return np.outer(v, v.conj())


def add_ionq_native_bellpair_psi_minus(qc: QuantumCircuit, a: int, b: int):
    """
    Prepare |Psi^->_ab using IonQ native gates only.
    This is local syntax-only use: no provider, no cloud auth.
    """
    # RY(+pi/2) on control-like qubit
    qc.append(GPI2Gate(0.25), [a])

    # CNOT-style trapped-ion decomposition pieces
    qc.append(GPI2Gate(0.25), [a])
    qc.append(MSGate(0.0, 0.0), [a, b])
    qc.append(GPI2Gate(-0.5), [a])
    qc.append(GPI2Gate(-0.5), [b])
    qc.append(GPI2Gate(-0.25), [a])


def main():
    os.makedirs("results", exist_ok=True)

    # Build two native Bell-pair resources: |Psi^->_01 tensor |Psi^->_23
    qc = QuantumCircuit(4)
    add_ionq_native_bellpair_psi_minus(qc, 0, 1)
    add_ionq_native_bellpair_psi_minus(qc, 2, 3)

    psi = Statevector.from_instruction(qc).data

    # Bell state |Phi+>
    phi_plus = (np.kron(ket0(), ket0()) + np.kron(ket1(), ket1())) / np.sqrt(2.0)
    bell_proj = projector(phi_plus)

    # Project q1,q2 onto |Phi+>
    P12 = (
        np.kron(projector(ket0()), np.kron(bell_proj, projector(ket0())))
        + np.kron(projector(ket0()), np.kron(bell_proj, projector(ket1())))
        + np.kron(projector(ket1()), np.kron(bell_proj, projector(ket0())))
        + np.kron(projector(ket1()), np.kron(bell_proj, projector(ket1())))
    )

    psi_post_unnorm = P12 @ psi
    success_probability = float(np.vdot(psi_post_unnorm, psi_post_unnorm).real)

    if success_probability < 1e-15:
        raise RuntimeError("Postselected branch has zero probability.")

    psi_post = psi_post_unnorm / np.sqrt(success_probability)
    rho_post = DensityMatrix(np.outer(psi_post, psi_post.conj()))

    # Trace out q1 and q2, keep q0 and q3
    rho_03 = partial_trace(rho_post, [1, 2])

    target_rho = DensityMatrix(np.outer(phi_plus, phi_plus.conj()))
    bell_fidelity = float(state_fidelity(rho_03, target_rho))
    entanglement_yield = success_probability * bell_fidelity

    result = {
        "stack": "ionq_native",
        "paradigm": "DV",
        "protocol": "two_bellpair_entanglement_swapping",
        "success_probability": success_probability,
        "bell_fidelity": bell_fidelity,
        "entanglement_yield": entanglement_yield,
        "notes": "Local IonQ-native syntax using GPI2/MS only; no cloud submission."
    }

    with open("results/ionq_native.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

    print(qc)
    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()

     ┌────────────┐┌────────────┐┌───────────────┐┌────────────┐┌─────────────┐
q_0: ┤ Gpi2(0.25) ├┤ Gpi2(0.25) ├┤0              ├┤ Gpi2(-0.5) ├┤ Gpi2(-0.25) ├
     └────────────┘└────────────┘│  Ms(0,0,0.25) │├────────────┤└─────────────┘
q_1: ────────────────────────────┤1              ├┤ Gpi2(-0.5) ├───────────────
     ┌────────────┐┌────────────┐├───────────────┤├────────────┤┌─────────────┐
q_2: ┤ Gpi2(0.25) ├┤ Gpi2(0.25) ├┤0              ├┤ Gpi2(-0.5) ├┤ Gpi2(-0.25) ├
     └────────────┘└────────────┘│  Ms(0,0,0.25) │├────────────┤└─────────────┘
q_3: ────────────────────────────┤1              ├┤ Gpi2(-0.5) ├───────────────
                                 └───────────────┘└────────────┘               
{
  "stack": "ionq_native",
  "paradigm": "DV",
  "protocol": "two_bellpair_entanglement_swapping",
  "success_probability": 0.2499999999999995,
  "bell_fidelity": 0.9999999999999991,
  "entanglement_yield": 0.24999999999999928,
  "notes": "Local IonQ-native syntax using GPI2/MS 